In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","appointments","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:

# appointment_id
df_silver = df_silver.withColumn(
    "appointment_id",
    F.trim(F.col("appointment_id"))
# resident_id
).withColumn(
    "resident_id",
    F.trim(F.col("resident_id"))
# employee_id
).withColumn(
    "employee_id",
    F.trim(F.col("employee_id"))
# appointment_date
).withColumn(
    "appointment_date",
    F.trim(F.col("appointment_date"))
# appointment_type
).withColumn(
    "appointment_type",
    F.trim(F.col("appointment_type"))
# status
).withColumn(
    "status",
    F.trim(F.col("status"))
# location
).withColumn(
    "location",
    F.trim(F.col("location"))
# notes
).withColumn(
    "notes",
    F.trim(F.col("notes"))
# created_at
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns])
display(null_count)

#### Cleaning data in table

In [0]:
# appointment_id
from pyspark.sql.functions import col,when,abs,trim,upper,initcap

dup = df_silver.filter(col("appointment_id").rlike("^//APT"))
display(dup)

In [0]:
# resident_id
from pyspark.sql.functions import col,when,abs,trim,upper,initcap

dup = df_silver.filter(col("resident_id").rlike("^//APT"))
display(dup)

In [0]:
# employee_id
from pyspark.sql.functions import col,when,abs,trim,upper,initcap

dup = df_silver.filter(col("employee_id").rlike("^//APT"))
display(dup)


In [0]:
# appointment_date
from pyspark.sql.functions import col,when,abs,trim,upper,initcap,to_timestamp
dup = df_silver.groupBy("appointment_date").count().filter(col("count")>100)
# display(dup)

reply_dup = {"not-a-date" : None ,
"2030-01-01" : None ,
"32/13/2026" : None ,
"9999-99-99" : None ,
"2026-02-30" : None ,
"00/00/0000" : None,
"": None}
df_silver = df_silver.replace(reply_dup,subset = ["appointment_date"])

dup = df_silver.groupBy("appointment_date").count().filter(col("count")>100)
display(dup)
df_silver = df_silver.withColumn("appointment_date",to_timestamp(col("appointment_date"),"yyyy-MM-dd HH:mm:ss"))
display(df_silver)

In [0]:
# appointment_type

from pyspark.sql.functions import col,when,abs,trim,upper,initcap
df_silver = df_silver.withColumn("appointment_type",upper(trim(col("appointment_type"))))
dup = df_silver.groupBy("appointment_type").count().filter(col("count")>1)
# display(dup)


reply_xv = {"NAN" : "UNKNOWN" ,
            "#N/A" : "UNKNOWN" ,
"N/A" : "UNKNOWN" ,
"UNKNOWN" : "UNKNOWN" ,
"NULL" : "UNKNOWN" ,
"" : "UNKNOWN" ,
"NONE" : "UNKNOWN" ,
"null" : "UNKNOWN" }
df_silver = df_silver.replace(reply_xv,subset = ["appointment_type"])
df_silver = df_silver.fillna({"appointment_type":"UNKNOWN"})

dup = df_silver.groupBy("appointment_type").count().filter(col("count")>1)
display(dup)
# display(df_silver)

In [0]:
# status

from pyspark.sql.functions import col,when,abs,trim,upper,initcap
df_silver = df_silver.withColumn("status",initcap(trim(col("status"))))
dup = df_silver.groupBy("status").count().filter(col("count")>1)
# display(dup)

rpl_xc = {"N/a" : "Unknown",
"#n/a" : "Unknown",
"Nan" : "Unknown",
"Null" : "Unknown",
"None" : "Unknown",
"Unknown" : "Unknown",
"" : "Unknown"
}
df_silver = df_silver.replace(rpl_xc,subset = ["status"])
df_silver = df_silver.fillna({"status":"Unknown"})
dup = df_silver.groupBy("status").count().filter(col("count")>1)
display(dup)

In [0]:
# location

from pyspark.sql.functions import col,when,abs,trim,upper,initcap
df_silver = df_silver.withColumn("location",upper(trim(col("location"))))
dup = df_silver.groupBy("location").count().filter(col("count")>1)
# display(dup)

rpl_xc = {"" :"UNKNOWN",
"UNKNOWN" :"UNKNOWN",
"NAN" :"UNKNOWN",
"NONE" :"UNKNOWN",
"NULL" :"UNKNOWN",
"N/A" :"UNKNOWN",
"#N/A" :"UNKNOWN",
"null" :"UNKNOWN",}
df_silver = df_silver.replace(rpl_xc,subset = ["location"])
df_silver = df_silver.fillna({"location":"Unknown"})
dup = df_silver.groupBy("location").count().filter(col("count")>1)
display(dup)

In [0]:
# notes
from pyspark.sql.functions import col,when,abs,trim,upper,initcap
df_silver = df_silver.withColumn("notes",initcap(trim(col("notes"))))
dup = df_silver.groupBy("notes").count().filter(col("count")>1)
# display(dup)

reply_dg = {"Nan" : "Unknown",
"None" : "Unknown",
"N/a" : "Unknown",
"#n/a" : "Unknown",
"Unknown" : "Unknown",
"Null" : "Unknown",
"" : "Unknown"
}
df_silver = df_silver.replace(reply_dg,subset = ["notes"])
df_silver = df_silver.fillna({"notes":"Unknown"})
dup = df_silver.groupBy("notes").count().filter(col("count")>1)
display(dup)

In [0]:

# created_at

from pyspark.sql.functions import col,when,abs,trim,upper,initcap

# df_silver = df_silver.withColumn("notes",initcap(trim(col("notes"))))
dup = df_silver.groupBy("created_at").count().filter(col("count")>10)
display(dup)
df_silver = df_silver.withColumn("created_at",to_timestamp(col("created_at"),"yyyy-MM-dd HH:mm:ss"))
display(df_silver)


#### Silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")